# Forecasting Model Zoo

Wiki reference for [the forecasting model zoo](https://ml-viz-ruby.vercel.app/wiki/forecasting-model-zoo).

**The idea in one sentence.** Modern forecasting is a zoo of purpose-built architectures — residual basis-expansion (**N-BEATS**, **N-HiTS**), attention (**TFT**), all-MLP mixers (**TSMixer / TSMixerx**), and pretrained **foundation models** (**Chronos**) — that sit between classical ARIMA/Prophet and generic deep nets.

**What this notebook does.** It builds the mechanism that made **N-BEATS** famous — *doubly-residual stacking* — from scratch in NumPy. We use the **interpretable** variant (trend block + seasonality block with closed-form least-squares fits) so every number is reproducible and you can *see* each block specialise on what the previous one couldn't explain. We then point at the one-line `neuralforecast` / `chronos` APIs you'd use in practice.

> Runs on Colab (Python 3.11) with only `numpy` + `matplotlib`. Copy to Drive to keep edits.

## 0 — Setup

A tiny synthetic series with an obvious **trend** + **seasonality** so the two N-BEATS blocks have something clean to decompose.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('dark_background')
rng = np.random.default_rng(7)

L, H, SEASON = 48, 12, 12          # lookback, horizon, seasonal period
t_full = np.arange(L + H)
trend_true = 0.4 * t_full + 5.0
seas_true  = 3.0 * np.sin(2 * np.pi * t_full / SEASON)
series = trend_true + seas_true + rng.normal(0, 0.4, L + H)

lookback = series[:L]              # what the model sees
future   = series[L:]             # what it must forecast (held out)
print('lookback shape', lookback.shape, '| horizon', H)

## 1 — One N-BEATS block, from scratch

A **block** takes a residual signal, fits a basis, and emits two things:

- a **backcast** — its reconstruction of the *input* window (length `L`),
- a **forecast** — its contribution to the *output* window (length `H`).

In the interpretable variant the basis is fixed and the block just learns coefficients. We solve for those coefficients in closed form with least squares (a real N-BEATS uses an MLP, but the input/output contract is identical). The basis is evaluated over the **joint** time grid so it extrapolates cleanly from the backcast region into the forecast region.

In [ ]:
def trend_basis(times, degree=2):
    """Polynomial basis over normalised time; shape (len(times), degree+1)."""
    tau = times / L                      # backcast in [0,1), forecast beyond 1
    return np.stack([tau ** p for p in range(degree + 1)], axis=1)

def seasonal_basis(times, n_harmonics=3, period=SEASON):
    """Fourier basis (cos+sin harmonics); shape (len(times), 2*n_harmonics)."""
    cols = []
    for k in range(1, n_harmonics + 1):
        cols.append(np.cos(2 * np.pi * k * times / period))
        cols.append(np.sin(2 * np.pi * k * times / period))
    return np.stack(cols, axis=1)

def nbeats_block(residual, basis_fn):
    """Fit basis coeffs on the backcast region, then emit backcast + forecast."""
    B_all = basis_fn(t_full)             # (L+H, k)
    B_bc, B_fc = B_all[:L], B_all[L:]    # split into backcast / forecast rows
    coeffs, *_ = np.linalg.lstsq(B_bc, residual, rcond=None)   # 'learned' weights
    backcast = B_bc @ coeffs             # (L,)
    forecast = B_fc @ coeffs             # (H,)
    return backcast, forecast

## 2 — Doubly-residual stacking

Chain the blocks. Each block subtracts its **backcast** from the running input before handing the remainder to the next block, and all **forecasts** are summed:

$$\mathbf{x}_\ell = \mathbf{x}_{\ell-1} - \hat{\mathbf{b}}_\ell, \qquad \hat{\mathbf{y}} = \sum_\ell \hat{\mathbf{f}}_\ell$$

Block 1 grabs the **trend**; block 2 only ever sees the leftover **seasonality**. Neither has to re-learn the other's job — exactly like boosting on residuals.

In [ ]:
residual = lookback.copy()
forecast_total = np.zeros(H)
block_specs = [('trend', trend_basis), ('seasonality', seasonal_basis)]

for name, basis_fn in block_specs:
    bc, fc = nbeats_block(residual, basis_fn)
    residual = residual - bc            # pass the remainder forward
    forecast_total = forecast_total + fc
    print(name.ljust(12), '| residual RMS after block =', round(float(np.sqrt(np.mean(residual**2))), 3))

mae = float(np.mean(np.abs(forecast_total - future)))
print('\nforecast MAE vs held-out future:', round(mae, 3))

The residual RMS **shrinks** block by block: the trend block removes the ramp, leaving a residual that is almost pure sine, which the seasonality block then absorbs. That monotone shrink is the signature of a working doubly-residual stack.

## 3 — Exact worked trace (matches the wiki page)

The same telescoping, on the hand-computed 2-block example from the wiki, verified to the digit.

In [ ]:
x0 = np.array([10, 12, 14, 16], dtype=float)
b1, f1 = np.array([9, 11.5, 14, 16.5]), np.array([18, 20])
b2, f2 = np.array([1, 0.5, 0, -0.5]),   np.array([-0.5, -1])

x1 = x0 - b1                 # remainder after block 1
x2 = x1 - b2                 # remainder after block 2 -> zero
y_hat = f1 + f2             # aggregate forecast

assert np.allclose(x1, [1, 0.5, 0, -0.5])
assert np.allclose(x2, [0, 0, 0, 0]), 'residual must telescope to zero'
assert np.allclose(y_hat, [17.5, 19.0])
print('residual after block 1:', x1)
print('residual after block 2:', x2, '(fully explained)')
print('aggregate forecast    :', y_hat)

## 4 — Visualize the decomposition

Left: each block's backcast reconstruction. Right: the summed forecast against the held-out future.

In [ ]:
# recompute per-block pieces for plotting
res = lookback.copy()
bc_trend, fc_trend = nbeats_block(res, trend_basis); res = res - bc_trend
bc_seas,  fc_seas  = nbeats_block(res, seasonal_basis)

fig, (axL, axR) = plt.subplots(1, 2, figsize=(12, 4))

axL.plot(np.arange(L), lookback, color='#94a3b8', lw=1.5, label='lookback')
axL.plot(np.arange(L), bc_trend, color='#6366f1', lw=2, label='block 1: trend backcast')
axL.plot(np.arange(L), bc_trend + bc_seas, color='#2dd4bf', lw=2, ls='--',
         label='block 1+2: full reconstruction')
axL.set_title('Backcast: blocks reconstruct the input'); axL.legend(fontsize=8)

fc_total = fc_trend + fc_seas
axR.plot(np.arange(L), lookback, color='#94a3b8', lw=1.5, label='lookback')
axR.plot(np.arange(L, L + H), future, color='#f43f5e', lw=2, marker='o', ms=3, label='actual future')
axR.plot(np.arange(L, L + H), fc_total, color='#facc15', lw=2, marker='s', ms=3, label='N-BEATS forecast')
axR.axvline(L - 0.5, color='#475569', ls=':')
axR.set_title('Forecast: sum of block forecasts'); axR.legend(fontsize=8)

plt.tight_layout(); plt.show()

**What to notice.** The trend backcast (indigo) captures the ramp; adding the seasonality block (teal dashed) snaps the reconstruction onto the wiggly input. On the right, the summed forecast (yellow) tracks the held-out future (rose) because each future component was extrapolated from the *same* fitted basis — trend keeps rising, seasonality keeps oscillating.

## 5 — The library way

You would never hand-roll this in production. Nixtla's **`neuralforecast`** ships N-BEATS, N-HiTS, TFT, TSMixer, and TSMixerx behind one API; **`chronos-forecasting`** gives you Amazon's pretrained, **zero-shot** foundation model. These need installs + (for Chronos) a model download, so they are shown as reference rather than executed here.

```python
# pip install neuralforecast
from neuralforecast import NeuralForecast
from neuralforecast.models import NBEATS, NHITS, TFT, TSMixerx

nf = NeuralForecast(
    models=[
        NHITS(h=12, input_size=48, max_steps=500),      # multi-rate residual stacks
        TFT(h=12, input_size=48, max_steps=500),        # attention + variable selection
        TSMixerx(h=12, input_size=48, n_series=1, max_steps=500),  # all-MLP mixer + covariates
    ],
    freq='H',
)
nf.fit(df)              # df columns: unique_id, ds, y (+ any covariates)
preds = nf.predict()    # one row per series per horizon step

# --- Zero-shot foundation model: no fitting at all ---
# pip install chronos-forecasting
import torch
from chronos import ChronosPipeline

pipe = ChronosPipeline.from_pretrained('amazon/chronos-t5-small', device_map='cpu')
ctx = torch.tensor(lookback)                 # any brand-new series
samples = pipe.predict(ctx, prediction_length=12)   # (num_samples, 12) trajectories
median = np.median(samples[0].numpy(), axis=0)      # point forecast
```

## 6 — Tradeoffs & when to use each

| Model | Core idea | Covariates | Best when |
|---|---|---|---|
| **ARIMA/SARIMA** | Linear AR + MA on differenced series | ARIMAX | Short univariate, exact intervals |
| **Prophet** | Additive trend + Fourier season + holidays | Yes | Calendar seasonality, non-experts |
| **N-BEATS** | Doubly-residual FC blocks | N-BEATSx | Strong univariate neural baseline |
| **N-HiTS** | Multi-rate pooling + hierarchical interp. | Yes | **Long** horizons, cheap |
| **TFT** | Attention + variable selection | Rich (static/future) | Interpretability + covariates |
| **TSMixer(x)** | Alternating time/feature MLPs | TSMixerx | Multivariate, long horizon, cheap |
| **Chronos** | Tokenise values -> pretrained LM | Model-dep. | **Zero-shot** / cold-start |

**Failure modes.** Leaking future-unknown covariates into TFT/TSMixerx as 'known-future'; forgetting to scale per-series before a global model; trusting Prophet's uncalibrated bands; judging a foundation model on one series. Always compare with walk-forward validation, never random splits.

## ✏️ Your turn — add a third (constant/level) block

Real stacks have many blocks. Add a **level** block *before* the trend block whose basis is just a constant (a single column of ones). Fit it first, subtract its backcast, then run trend + seasonality on the remainder. Does the final MAE change much? (It shouldn't — the trend block's degree-0 term already absorbs a constant. That redundancy is why real N-BEATS lets blocks *learn* their bases instead of fixing them.)

In [ ]:
def level_basis(times):
    # TODO(you): return a constant basis of shape (len(times), 1) -- a column of ones
    raise NotImplementedError

# res = lookback.copy()
# forecast = np.zeros(H)
# for basis_fn in [level_basis, trend_basis, seasonal_basis]:
#     bc, fc = nbeats_block(res, basis_fn)
#     res = res - bc
#     forecast = forecast + fc
# print('MAE with level block:', round(float(np.mean(np.abs(forecast - future))), 3))

<details><summary>Solution</summary>

```python
def level_basis(times):
    return np.ones((len(times), 1))

res = lookback.copy()
forecast = np.zeros(H)
for basis_fn in [level_basis, trend_basis, seasonal_basis]:
    bc, fc = nbeats_block(res, basis_fn)
    res = res - bc
    forecast = forecast + fc
print('MAE with level block:', round(float(np.mean(np.abs(forecast - future))), 3))
```

The MAE is essentially unchanged: the constant is already representable by the trend basis, so the extra block contributes almost nothing after the least-squares fit.
</details>

## Key takeaways

- **Doubly-residual stacking** is the N-BEATS engine: each block reconstructs (backcasts) part of the input, subtracts it, and the leftover forces the next block to specialise — boosting learned end-to-end.
- The **backcast** is not wasted output; it is what makes the residual meaningful and drives the additive decomposition.
- **N-HiTS** extends this with multi-rate pooling for cheap long-horizon forecasts; **TFT** swaps in attention + variable selection for interpretability; **TSMixer(x)** shows plain MLPs often suffice; **Chronos** skips training entirely with a pretrained LM over tokenised values.
- Pick with data volume, covariates, horizon length, and interpretability needs in mind — and always judge them with **walk-forward validation**.

**Next:** [Deep Learning for Time Series](https://ml-viz-ruby.vercel.app/courses/time-series/03-deep-learning-for-time-series) · [Hierarchical Forecasting](https://ml-viz-ruby.vercel.app/wiki/hierarchical-forecasting)